# C12-classical-models — Practice p26 — Solution


The stable trace and the written proof separately certify perfect classification, persistent likelihood pressure, and the absence of a finite unregularized minimizer.


In [ ]:
import numpy as np

x_p26 = np.array([-3., -2., -1., 1., 2., 3.], dtype=np.float64)
y_p26 = np.array([0., 0., 0., 1., 1., 1.], dtype=np.float64)
w_values_p26 = np.array([0.5, 1., 2., 4., 8., 16.], dtype=np.float64)


def separation_trace(w_values):
    values = np.asarray(w_values)
    if values.dtype != np.float64 or values.ndim != 1 or values.shape != (6,) or not np.isfinite(values).all():
        raise ValueError("w_values must be the finite float64 six-vector")
    logits = values[:,None] * x_p26[None,:]
    probabilities = np.empty_like(logits)
    nonnegative = logits >= 0.0
    probabilities[nonnegative] = 1.0/(1.0+np.exp(-logits[nonnegative]))
    exponentials = np.exp(logits[~nonnegative])
    probabilities[~nonnegative] = exponentials/(1.0+exponentials)
    losses = (np.maximum(logits,0.0)-y_p26*logits+np.log1p(np.exp(-np.abs(logits)))).mean(axis=1).astype(np.float64)
    gradients = (((probabilities-y_p26)*x_p26).mean(axis=1)).astype(np.float64)
    predictions = (probabilities >= 0.5).astype(np.int64)
    accuracies = np.mean(predictions == y_p26.astype(np.int64), axis=1).astype(np.float64)
    return {"losses":losses,"probabilities":probabilities.astype(np.float64),
            "gradients":gradients,"predictions":predictions,"accuracies":accuracies}


trace_p26 = separation_trace(w_values_p26)
analysis_p26 = '''For w>0, negative x has probability below one half and positive x has probability above one half, so every row is correct. Pairing plus/minus a gives L(w)=(1/3) sum over a=1,2,3 of log(1+exp(-a*w)), whose derivative is -(1/3) sum a/(1+exp(a*w))<0. Each finite term is positive while all approach zero as w grows, so zero is an unattained infimum. The trace certifies accuracy one, strictly decreasing loss, and nonzero negative gradients. L2 regularization or a finite parameter/stopping constraint remedies divergence. Select regularization on development folds; then audit held-out BCE, Brier score, and reliability, optionally using a disjoint calibration fold, before one locked test evaluation.'''


### Answer check


In [ ]:
ATOL=1e-12
RTOL=1e-10
expected_losses_p26=np.array([0.32958398322702731,0.16292568337831248,0.049184541366170891,0.0061638261613944133,0.00011183964860516716,3.7511727017114599e-08])
expected_gradients_p26=np.array([-0.48756669431906824,-0.21654162831564347,-0.054197737138734957,-0.0062251142489436785,-0.00011185843801487086,-3.7511729129065191e-08])
assert set(trace_p26)=={"losses","probabilities","gradients","predictions","accuracies"}
assert np.allclose(trace_p26["losses"],expected_losses_p26,atol=ATOL,rtol=RTOL)
assert np.allclose(trace_p26["gradients"],expected_gradients_p26,atol=ATOL,rtol=RTOL)
assert np.allclose(trace_p26["accuracies"],np.ones(6),atol=ATOL,rtol=RTOL)
assert np.all(np.diff(trace_p26["losses"])<0.0) and np.all(trace_p26["gradients"]<0.0)
assert isinstance(analysis_p26,str) and "unattained" in analysis_p26
